# Sort and Plot Tradeoffs

Notebook conversion of `example_scripts/sort_and_plot_tradeoffs.py`.

The original script expects `AllPolicies.csv`. This notebook preserves that logic and makes the missing-input condition explicit.

## Context Files in data/

Copied from real project folders for reference:
- `Plot.HiPlot.csv`
- `redriver_v1.8.3.xml`
- `RedRiver-HaydenCarsonExample.v3.mdl.gz`

If you have the original `AllPolicies.csv`, place it in `data/AllPolicies.csv`.

In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns
import plotly.express as px

from borgRWhelper.tradeoff import parallel_plot_hp, label_eps_nd

In [ ]:
all_policies_path = Path("data/AllPolicies.csv")
output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)

if not all_policies_path.exists():
    print("Missing required input: data/AllPolicies.csv")
    print("A real context file is available as data/Plot.HiPlot.csv, but it uses a different schema than this legacy script.")
else:
    # import archive (in 'readable' format)
    all_solutions_df = pd.read_csv(all_policies_path, delimiter=",")

    objective_names = ["Objectives.Powell_3490",
                       "Objectives.Powell_WY_Release",
                       "Objectives.Mead_1000",
                       "Objectives.LB_Shortage_Volume"]
    num_objs = len(objective_names)

    # directions for each objective (must be 'minimize' or 'maximize')
    # TODO: we're not positive that these are correct for this example, since
    # the data file that this refers to seems to be missing
    objective_directions = ["maximize",
                            "maximize",
                            "maximize",
                            "minimize"]

    metric_names = ["Objectives.LB_Shortage_Frequency",
                    "Objectives.Mead_1020",
                    "Objectives.Powell_3525",
                    "Objectives.Powell_Release_LTEMP",
                    "Objectives.Start_in_EQ",
                    "Objectives.VariableHydrologicShortageIndicatorMetric",
                    "Objectives.Avg_Powell_PE",
                    "Objectives.Lee_Ferry_Deficit",
                    "Objectives.Avg_Mead_PE",
                    "Objectives.Max_Delta_Annual_Shortage",
                    "Objectives.Avg_Annual_LB_Policy_Shortage"]

    epsilons = [1,
                50000,
                1,
                50000]

    all_solutions_df = label_eps_nd(all_solutions_df, "Eps Nd", objective_names, objective_directions, epsilons)

    # save individual dataframes from each experiment
    exp1_solutions_df = all_solutions_df.loc[all_solutions_df["Experiment"] == 1].copy(deep=True)
    exp2_solutions_df = all_solutions_df.loc[all_solutions_df["Experiment"] == 2].copy(deep=True)
    exp3_solutions_df = all_solutions_df.loc[all_solutions_df["Experiment"] == 3].copy(deep=True)
    exp4_solutions_df = all_solutions_df.loc[all_solutions_df["Experiment"] == 4].copy(deep=True)

    # create parallel plots and save into html
    parallel_plot_hp(all_solutions_df[["Experiment",
                                       "Mead_Shortage_V_DV Row 0"] + metric_names + objective_names],
                     objective_names, objective_directions).to_html(output_dir / "all_solutions_with-metrics.html")
    parallel_plot_hp(exp1_solutions_df[["Mead_Shortage_V_DV Row 0"] + metric_names + objective_names],
                     objective_names, objective_directions).to_html(output_dir / "exp1_solutions_with-metrics.html")
    parallel_plot_hp(exp2_solutions_df[["Mead_Shortage_V_DV Row 0"] + metric_names + objective_names],
                     objective_names, objective_directions).to_html(output_dir / "exp2_solutions_with-metrics.html")
    parallel_plot_hp(exp3_solutions_df[["Mead_Shortage_V_DV Row 0"] + metric_names + objective_names],
                     objective_names, objective_directions).to_html(output_dir / "exp3_solutions_with-metrics.html")
    parallel_plot_hp(exp4_solutions_df[["Mead_Shortage_V_DV Row 0"] + metric_names + objective_names],
                     objective_names, objective_directions).to_html(output_dir / "exp4_solutions_with-metrics.html")

    # heatmap of correlation
    sns_plot = sns.clustermap(exp2_solutions_df[metric_names + objective_names].corr(), cmap="rocket_r")

    fig = px.scatter_matrix(exp2_solutions_df[objective_names])
    fig.show()

    fig = px.scatter_matrix(exp3_solutions_df[objective_names])
    fig.show()

    print(f"Wrote HTML outputs to: {output_dir}")